# Data Versioning & Data Lineage

Let's study how we could go about using and tracking data during a data science project.

Lots of tools are usable for this purpose. We will use [DVC](https://dvc.org/) for this demonstration.

In [ ]:
# @title ## Environment setup
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!apt install tree
!pip install dvc

## Initializing the project repository

In [ ]:
!git init
!dvc init

## Getting some data

There would be several ways to get some data for this demonstration into the project repository. We will use [`dvc get`](https://dvc.org/doc/command-reference/get) here. We will then track it with [`dvc add`](https://dvc.org/doc/command-reference/add).

In [ ]:
!dvc get https://github.com/m09/dataset-wikipedia-movie-plots wiki_movie_plots_deduped.csv -o data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc

## Storing metadata in the project repository

In [ ]:
!git add .gitignore data.csv.dvc

In [ ]:
!git commit -m "Ajout d'une première version des données"
!git tag "v1"

## Storing data in a data storage server

With the DVC tool, data is stored separately from metadata. Here we'll just setup a fake data remote on this computer, but usually it will be a dedicated server (on AWS for example).

In [ ]:
!dvc remote add -d local /tmp/dvcstore

In [ ]:
!dvc push

## Modifying data and tracking changes

In [ ]:
!sort -r < data.csv > a && dvc remove data.csv.dvc && mv a data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc

In [ ]:
!git add data.csv.dvc .gitignore

In [ ]:
!git commit -m "Données v2"
!git tag "v2"

In [ ]:
!dvc push

## Example of versioned data usage: rolling back

Let's say we have a problem with the new version of the data. We can then go back to the first version easily: that's the big selling point of versioning data.

In [ ]:
!git checkout v1 data.csv.dvc

In [ ]:
!cat data.csv.dvc
!tree .dvc

In [ ]:
!dvc checkout data.csv

We can now store the metadata that specifies that we rolled back in our repository.

In [ ]:
!git add data.csv.dvc

In [ ]:
!git commit -m "Retour aux données v1"
!git tag v3

In [ ]:
!dvc push

## Data processing

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

Path(argv[2]).write_text(
    Path(argv[1]).read_text(encoding="utf8").upper(),
    encoding="utf8")

We can use [`dvc stage add`](https://dvc.org/doc/command-reference/stage/add) to add a stage of processing to our metadata system. This way we'll know exactly what we have processed and the results we obtained.

In [ ]:
!dvc stage add -n transform-uppercase -d data.csv -o data-upper.csv python upper.py data.csv data-upper.csv

[`dvc repro`](https://dvc.org/doc/command-reference/repro) will execute this step.

In [ ]:
!dvc repro

In [ ]:
!git add dvc.yaml .gitignore dvc.lock

In [ ]:
!dvc push